In [11]:
import onnx
import hls4ml
from qonnx.util.cleanup import cleanup
from qonnx.transformation.general import GiveUniqueNodeNames
from qonnx.util.to_channels_last import to_channels_last



In [ ]:
import onnx
import hls4ml
from onnxsim import simplify
from qonnx.util.cleanup import cleanup
from qonnx.util.to_channels_last import to_channels_last

input_file = 'mnist_finn_ready.onnx'
channels_last_file = 'mnist_channels_last.onnx'

# 1. Transform to Channels Last (as before)
print("Converting to channels-last...")
cleanup(input_file, out_file='temp.onnx')
to_channels_last('temp.onnx', out_file=channels_last_file, make_input_channels_last=True)

# 2. Simplify and Fold Gemm
# Now that names are safe, simplification should not crash.
# This converts Gemm into MatMul + Add, which hls4ml prefers.
print("Simplifying graph (Folding Gemm -> MatMul)...")
model_obj = onnx.load(channels_last_file)
model_simple, check = simplify(model_obj)
assert check, "Simplified ONNX model could not be validated"

# 3. Final Name Sanitization
# Ensure no empty names or weird characters remain after simplification
for i, node in enumerate(model_simple.graph.node):
    node.name = f"{node.op_type}_{i}"

# 4. Convert to HLS
print("Generating hls4ml config...")
config = hls4ml.utils.config_from_onnx_model(
    model_simple,
    granularity='name',
    backend='Vivado'
)

# Optional: Force the precision for the whole model if it's a quantized FINN model
# config['Model']['Precision'] = 'ap_fixed<16,6>'

hls_model = hls4ml.converters.convert_from_onnx_model(
    model_simple,
    hls_config=config,
    output_dir='my_hls_project',
    fpga_part='xc7z020clg484-1'
)

print("Success! HLS project generated.")

Converting to channels-last...
Simplifying graph (Folding Gemm -> MatMul)...
Generating hls4ml config...
Output layers:  ['Gemm_21']
Input shape: [28, 28, 1]
Topology:


RuntimeError: Could not find the shape for input Quant_1_param0